# Limpieza y ingeniería de atributos
Este notebook va a ser para limpiar el dataframe.

Se recibe como entrada un dataframe (puede ser el de cualquier año o el consolidado total) y se van a procesar y limpiar las siguientes columnas:

- Género: Poner únicamente los géneros en F, M, No especificado
- Edad: Verificar que no haya edades negativas o que no cuadren (muy bajas o muy altas)
- Fechas: Verificar que estén en formato DD/MM/AAAA
- Eliminar nulos y duplicados

# Bibliotecas

In [1]:
%pip install polars

Note: you may need to restart the kernel to use updated packages.


In [2]:
import polars as pl
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import re
from collections import Counter

# Obtenemos los datos de entrada

In [ ]:
df = pl.read_parquet("G:/.shortcut-targets-by-id/1NDRK_eUkNH2l92aHtOZVY-QThQVb6lil/Tlamatini/DATA/consolidado_por_año/consolidado_2024.parquet")
df


Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
str,f64,i64,str,str,str,str,str,str,str
"""M""",48.0,3371879,"""113""","""31/12/2023""","""23:21:52""","""659""","""01/01/2024""","""00:00:02""","""2024/1.csv"""
"""F""",26.0,7033434,"""281""","""31/12/2023""","""23:46:02""","""047""","""01/01/2024""","""00:00:02""","""2024/1.csv"""
"""F""",37.0,7169857,"""015""","""31/12/2023""","""23:51:57""","""217""","""01/01/2024""","""00:00:49""","""2024/1.csv"""
"""F""",30.0,6368211,"""555""","""31/12/2023""","""23:53:10""","""008""","""01/01/2024""","""00:02:39""","""2024/1.csv"""
"""M""",30.0,5136924,"""555""","""31/12/2023""","""23:52:53""","""008""","""01/01/2024""","""00:02:45""","""2024/1.csv"""
…,…,…,…,…,…,…,…,…,…
"""M""",31.0,8088551,"""425""","""30/09/2024""","""23:49:18""","""371""","""30/09/2024""","""23:59:19""","""2024/9.csv"""
"""F""",31.0,7701659,"""107-108""","""30/09/2024""","""23:48:06""","""006""","""30/09/2024""","""23:59:23""","""2024/9.csv"""
"""M""",23.0,2022929,"""004""","""30/09/2024""","""23:44:19""","""505""","""30/09/2024""","""23:59:37""","""2024/9.csv"""


# Eliminamos nulos y duplicados

In [145]:
# Verificar cuántos nulos hay
df.null_count()

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
93,662,0,0,0,0,0,0,0,0


In [20]:
# Rellenar con "No especificado" todos los nulos existentes
df = df.fill_null("No especificado")

In [21]:
# Verificamos que ya no haya nulos
df.null_count()

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0


In [36]:
# Ver cuántos duplicados hay
df.is_duplicated().sum()

0

In [23]:
#Para eliminar duplicados
df.unique()

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
str,str,str,str,str,str,str,str,str,str
"""M""","""50""","""3513""","""29""","""2016-02-08""","""13:11:43.213000""","""71""","""2016-02-08""","""13:38:00.420000""","""2016/2.csv"""
"""M""","""52""","""1716""","""168""","""2016-04-11""","""08:25:42.317000""","""182""","""2016-04-11""","""08:29:29""","""2016/4.csv"""
"""M""","""45""","""2297""","""6""","""2016-05-31""","""15:49:22.623000""","""313""","""2016-05-31""","""16:16:59""","""2016/5.csv"""
"""F""","""59""","""4899""","""187""","""2016-06-21""","""12:38:23.020000""","""187""","""2016-06-21""","""12:57:55""","""2016/6.csv"""
"""M""","""36""","""4440""","""396""","""2016-01-06""","""19:33:56.523000""","""425""","""2016-01-06""","""19:40:23""","""2016/1.csv"""
…,…,…,…,…,…,…,…,…,…
"""M""","""29""","""1869""","""43""","""2016-01-06""","""10:01:23.867000""","""26""","""2016-01-06""","""10:17:01""","""2016/1.csv"""
"""M""","""30""","""2544""","""106""","""2016-05-26""","""09:50:46.310000""","""19""","""2016-05-26""","""10:11:18""","""2016/5.csv"""
"""M""","""36""","""4992""","""194""","""2016-05-27""","""14:31:34.263000""","""52""","""2016-05-27""","""14:50:13""","""2016/5.csv"""


In [59]:
# Ver cuántos duplicados hay
df.is_duplicated().sum()

0

# Tratamiento a columna "Genero_Usuario"
Solo vamos a tomar en cuenta los registros que sean 'F', 'M' y 'No especificado'. Los registros que no sean 'F' o 'M', serán reemplazados por 'No especificado'.

Estos registros ya son conocidos, son los valores: 'O', '?' y 'nan'

In [25]:
valores_no_estandar = ['O', '?', 'nan'] 
df = df.with_columns(
    pl.col("Genero_Usuario").replace(valores_no_estandar, 'No especificado')
)

print("\nValores de 'Genero_Usuario' después de la limpieza:")
print(df['Genero_Usuario'].value_counts())


Valores de 'Genero_Usuario' después de la limpieza:
shape: (2, 2)
┌────────────────┬─────────┐
│ Genero_Usuario ┆ count   │
│ ---            ┆ ---     │
│ str            ┆ u32     │
╞════════════════╪═════════╡
│ M              ┆ 6810462 │
│ F              ┆ 2259081 │
└────────────────┴─────────┘


# Tratamiento a columnas de fechas

In [13]:
#Revisar si podemos estandarizar los años con horas extrañas, sino, solo trabajamos desde 2017
df = df.with_columns(
    pl.col("Fecha_Retiro")
      .cast(pl.Utf8)
      .str.replace("-", "/")
      .str.strptime(pl.Date, format="%d/%m/%Y", strict=False),

    pl.col("Fecha_Arribo")
      .cast(pl.Utf8)
      .str.replace("-", "/")
      .str.strptime(pl.Date, format="%d/%m/%Y", strict=False)
)

print("Procesado correctamente")

Procesado correctamente


In [26]:
# Verificamos que ya no haya nulos
df.null_count()

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0


In [74]:
df.select(
    pl.col("Hora_Retiro").is_null().sum().alias("Nulos_Nativos_R"),
    (pl.col("Hora_Retiro") == pl.lit("")).sum().alias("Cadenas_Vacias_R"),
    (pl.col("Hora_Retiro") == pl.lit("...")).sum().alias("Otros_Invalidos_R"),
    pl.col("Hora_Arribo").is_null().sum().alias("Nulos_Nativos_A"),
    (pl.col("Hora_Arribo") == pl.lit("")).sum().alias("Cadenas_Vacias_A"),
    (pl.col("Hora_Arribo") == pl.lit("...")).sum().alias("Otros_Invalidos_A")
)

Nulos_Nativos_R,Cadenas_Vacias_R,Otros_Invalidos_R,Nulos_Nativos_A,Cadenas_Vacias_A,Otros_Invalidos_A
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


In [ ]:
# Normalizador hora

def normalizar_hora(col):

    s = pl.col(col).str.strip_chars()

    # 1) Quitar microsegundos
    s = s.str.replace(r"\.\d+", "")

    # 2) Convertir AM/PM a 24h manualmente
    is_pm = s.str.contains("PM")
    is_am = s.str.contains("AM")

    # Extraer hh, mm, ss aunque tengan 1 solo dígito
    hh = s.str.extract(r"(\d+):", 1)
    mm = s.str.extract(r":(\d+):", 1)
    ss = s.str.extract(r":(\d+)\s?(AM|PM)?$", 1)

    # Convertir hh a entero
    hh_int = hh.cast(pl.Int32())

    # Reglas AM/PM
    hh_24 = (
        pl.when(is_pm & (hh_int != 12)).then(hh_int + 12)
        .when(is_pm & (hh_int == 12)).then(12)
        .when(is_am & (hh_int == 12)).then(0)
        .otherwise(hh_int)
    )

    # 3) Reconstruir HH:MM:SS con ceros
    final = (
        hh_24.cast(pl.Utf8()).str.zfill(2) + ":" +
        mm.str.zfill(2) + ":" +
        ss.str.zfill(2)
    )

    # 4) Validar como hora
    return (
        final
        .str.strptime(pl.Time, strict=False)
        .dt.strftime("%H:%M:%S")
        .alias(col)
    )



In [147]:
df = df.with_columns([
    normalizar_hora("Hora_Retiro"),
    normalizar_hora("Hora_Arribo")
])

df

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
str,f64,i64,str,str,str,str,str,str,str
"""M""",48.0,3371879,"""113""","""31/12/2023""","""23:21:52""","""659""","""01/01/2024""","""00:00:02""","""2024/1.csv"""
"""F""",26.0,7033434,"""281""","""31/12/2023""","""23:46:02""","""047""","""01/01/2024""","""00:00:02""","""2024/1.csv"""
"""F""",37.0,7169857,"""015""","""31/12/2023""","""23:51:57""","""217""","""01/01/2024""","""00:00:49""","""2024/1.csv"""
"""F""",30.0,6368211,"""555""","""31/12/2023""","""23:53:10""","""008""","""01/01/2024""","""00:02:39""","""2024/1.csv"""
"""M""",30.0,5136924,"""555""","""31/12/2023""","""23:52:53""","""008""","""01/01/2024""","""00:02:45""","""2024/1.csv"""
…,…,…,…,…,…,…,…,…,…
"""M""",31.0,8088551,"""425""","""30/09/2024""","""23:49:18""","""371""","""30/09/2024""","""23:59:19""","""2024/9.csv"""
"""F""",31.0,7701659,"""107-108""","""30/09/2024""","""23:48:06""","""006""","""30/09/2024""","""23:59:23""","""2024/9.csv"""
"""M""",23.0,2022929,"""004""","""30/09/2024""","""23:44:19""","""505""","""30/09/2024""","""23:59:37""","""2024/9.csv"""


In [148]:
# Verificamos que ya no haya nulos
df.null_count()

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
93,662,0,0,0,0,0,0,0,0
